In [ ]:
from sklearn.neighbors import KDTree
import pandas as pd

In [ ]:
fp =  "../data/sba_loans_prepared/sba_loans_num_enc_train.csv"
df_train = pd.read_csv(fp)

In [ ]:
preds = [ c for c in df_train.columns.tolist() if c != "LoanStatus"]

In [ ]:
from sklearn.neighbors import KDTree
X_train = df_train[preds]
kdt = KDTree(X_train, leaf_size=30, metric='euclidean')

In [ ]:
sel_bad_borr = df_train.LoanStatus == 1

In [ ]:
NUM_NBRS = 2
X_bad_borr_df = df_train[sel_bad_borr][preds]

In [ ]:
bdist, bind = kdt.query(X_bad_borr_df, k=NUM_NBRS)

In [ ]:
bind = bind.flatten().tolist() + X_bad_borr_df.index.tolist()

In [ ]:
df_bad_borr_nbrh = df_train[df_train.index.isin(bind)]

In [ ]:
df_bad_borr_nbrh.LoanStatus.value_counts()

In [ ]:
from sklearn.neighbors import NeighborhoodComponentsAnalysis

In [ ]:
nca = NeighborhoodComponentsAnalysis(random_state=42)

In [ ]:
X_df = df_bad_borr_nbrh[preds]
Y = df_bad_borr_nbrh["LoanStatus"]

In [ ]:
# this can take some time - a few minutes.
nca.fit(X_df, Y)

In [ ]:
X_trans_train = nca.transform(X_df)
df_train_nca_enc = pd.DataFrame(X_trans_train)
df_train_nca_enc.columns = ["nca-" + str(i+1) for i in range(df_train_nca_enc.shape[1])]
df_train_nca_enc["LoanStatus"] = Y.values

In [ ]:
fp =  "../data/sba_loans_prepared/sba_loans_nca_enc_train.csv"
df_train_nca_enc.to_csv(fp, index=False)

In [ ]:
fp =  "../data/sba_loans_prepared/sba_loans_num_enc_test.csv"
df_test = pd.read_csv(fp)

In [ ]:
X_test = df_test[preds]
X_trans_test = nca.transform(X_test)

In [ ]:
df_test_nca_enc = pd.DataFrame(X_trans_test)
nca_preds = ["nca-" + str(i+1) for i in range(df_test_nca_enc.shape[1])]
df_test_nca_enc.columns = nca_preds
df_test_nca_enc["LoanStatus"] = df_test["LoanStatus"].values
fp =  "../data/sba_loans_prepared/sba_loans_nca_enc_test.csv"
df_test_nca_enc.to_csv(fp, index=False)

In [ ]:
df_test_nca_enc.LoanStatus.value_counts()